# August 2026 forecast, step by step (graduate division)

Question 7 asks for an **August forecast for every Minnesota county**. The food-shelf file ends in **July 2026**. August 2026 has no reported numbers in this project, so the August 2026 columns are predictions.

The graduate submission has five numbers per county: **Visits** (`HouseholdsReg`), **Pounds** (`PoundsReg`), **Children** (`ChildrenReg`), **Adults** (`AdultsReg`), and **Seniors** (`SeniorsReg`). Individuals is also forecast here for the narrative; it is the sum of the three age groups.

The prediction used here, for each county and each measure:

**August 2026 = the average of May 2026, June 2026, and July 2026.**

Added across counties, the statewide forecast is **12,263,170 pounds**, **236,148 visits**, and **730,121 people** (258,020 children, 386,891 adults, 85,216 seniors).

All 87 counties are in the output. Wilkin has no reporting food shelf in the extract, so its forecast is zero rather than blank. The judges score with root mean square error (RMSE) across counties, so that is the score used to pick the method below.

August 2022, 2023, 2024, and 2025 are real months inside the file. They are used only to check the method. August 2026 is never read from anywhere.

Run this notebook from `analysis/Q7_august_forecast/`. It reads the cleaned extracts from `../EDA/cleaned/` (produced by `../EDA/EDA.ipynb`) and the organizers' template from `../../By Organizers/PredictionFile/`.


## Setup

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from clean_extracts import MN_COUNTIES

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "figure.figsize": (10, 4.2)})

## Step 1. Confirm the file ends in July 2026

`../EDA/cleaned/org_FSStats.csv` is one row per site per month. The measures we forecast are the ones the dictionary says to use:

- `PoundsReg` — pounds distributed
- `IndividualsReg` — people served that month (a person who comes twice is counted twice)
- `HouseholdsReg` — households served that month (the organizers call this **Visits**)
- `ChildrenReg`, `AdultsReg`, `SeniorsReg` — the age split of individuals; the graduate submission asks for these three instead of individuals

The last month in the file should be July 2026. If an August 2026 row appeared, we would stop.

In [2]:
fs = pd.read_csv("../EDA/cleaned/org_FSStats.csv", parse_dates=["DateForSearch"])

print(f"Rows: {len(fs):,}")
print(f"First month: {fs['DateForSearch'].min().date()}")
print(f"Last month:  {fs['DateForSearch'].max().date()}")
print("Months present in 2026:", sorted(fs.loc[fs['DateForSearch'].dt.year.eq(2026), 'DateForSearch'].dt.month.unique().tolist()))

if fs["DateForSearch"].max() != pd.Timestamp("2026-07-01"):
    raise RuntimeError("The extract does not end in July 2026. Stop and check the file before forecasting.")
if (fs["DateForSearch"] >= "2026-08-01").any():
    raise RuntimeError("August 2026 rows are in the extract. Do not use them.")

Rows: 28,243
First month: 2022-01-01
Last month:  2026-07-01
Months present in 2026: [1, 2, 3, 4, 5, 6, 7]


## Step 2. Add sites up to one number per county per month

A county can have many food shelves. The forecast is for the county, so each month we add the sites in that county together.

Minnesota has 87 counties. Wilkin has no organization and no stat rows. The submission template still has a Wilkin row, so the county table is widened to all 87 counties and a county-month with no reporting site is zero. That is also what the organizers' August actuals will show for Wilkin.

In [3]:
mn = fs[fs["County"].isin(MN_COUNTIES)].copy()
panel = (
    mn.groupby(["County", "DateForSearch"], as_index=False)
    .agg(
        pounds=("PoundsReg", "sum"),
        individuals=("IndividualsReg", "sum"),
        households=("HouseholdsReg", "sum"),
        children=("ChildrenReg", "sum"),
        adults=("AdultsReg", "sum"),
        seniors=("SeniorsReg", "sum"),
        sites=("_id_site", "nunique"),
    )
    .sort_values(["County", "DateForSearch"])
)

missing = sorted(set(MN_COUNTIES) - set(panel["County"]))
print(f"{panel['County'].nunique()} counties with data")
print(f"{panel['DateForSearch'].nunique()} months, {panel['DateForSearch'].min().date()} through {panel['DateForSearch'].max().date()}")
print("Counties with no rows (forecast will be zero):", missing)
panel.head(6)

86 counties with data
55 months, 2022-01-01 through 2026-07-01
Counties with no rows (forecast will be zero): ['Wilkin']


,County,DateForSearch,pounds,individuals,households,children,adults,seniors,sites
0,Aitkin,2022-01-01,"11,249.00",547.00,222.00,114.00,238.00,195.00,6
1,Aitkin,2022-02-01,"11,249.00",547.00,222.00,114.00,238.00,195.00,6
2,Aitkin,2022-03-01,"12,729.00",562.00,234.00,126.00,224.00,212.00,5
3,Aitkin,2022-04-01,"14,382.00",597.00,297.00,140.00,221.00,236.00,5
4,Aitkin,2022-05-01,"13,290.00",593.00,236.00,131.00,250.00,212.00,5
5,Aitkin,2022-06-01,"11,612.00",573.00,229.00,144.00,205.00,224.00,5


## Step 3. Look at past Augusts before choosing a formula

Past Augusts are in the file, so we can see the pattern. The question is whether August is usually a much bigger month than July.

In 2022 and 2023, August pounds were clearly above July. In 2024 and 2025, August pounds were about the same as July. A formula that always adds a big August jump would repeat a pattern that has already faded.

In [4]:
state = panel.groupby("DateForSearch")[["pounds", "individuals", "households"]].sum().sort_index()

rows = []
for year in range(2022, 2027):
    july = state.loc[pd.Timestamp(year, 7, 1)]
    august_stamp = pd.Timestamp(year, 8, 1)
    if august_stamp not in state.index:
        rows.append({"year": year, "july_pounds": july.pounds, "august_pounds": np.nan, "august_over_july": np.nan})
        continue
    august = state.loc[august_stamp]
    rows.append({
        "year": year,
        "july_pounds": july.pounds,
        "august_pounds": august.pounds,
        "august_over_july": august.pounds / july.pounds,
    })

season = pd.DataFrame(rows)
print("Statewide pounds. 2026 has no August row, so the ratio is blank.")
display(season.style.format({"july_pounds": "{:,.0f}", "august_pounds": "{:,.0f}", "august_over_july": "{:.3f}"}, na_rep="—"))

fig, ax = plt.subplots()
ax.plot(state.index, state["pounds"] / 1e6, color="#1D4E89", lw=2)
ax.axvline(pd.Timestamp("2026-07-01"), color="#B42318", ls="--", lw=1)
ax.set_title("Statewide pounds through July 2026")
ax.set_ylabel("Pounds (millions)")
ax.set_xlabel("Month")
fig.tight_layout()
plt.show()

Statewide pounds. 2026 has no August row, so the ratio is blank.


,year,july_pounds,august_pounds,august_over_july
0,2022,"9,675,162","10,885,356",1.125
1,2023,"11,286,715","13,000,256",1.152
2,2024,"13,293,611","13,235,462",0.996
3,2025,"12,796,557","12,821,574",1.002
4,2026,"12,999,407",—,—


/var/folders/jq/pnbcz95j78957hwkvq1nhmpr0000gn/T/ipykernel_86438/4012453668.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 4. Three forecasts we can explain

Each forecast is allowed to see months **up to July of that year**. It is not allowed to see that August.

1. **July only.** Predict August with that year's July.
2. **Last August.** Predict August with the previous August.
3. **May–July average.** Predict August with the average of May, June, and July of that year.

The next cell tries all three on August 2025, a month we already know, and prints the statewide pound total each one would have produced. This is a single-year illustration. The real choice uses every county, in the next step.

In [5]:
def wide(metric: str) -> pd.DataFrame:
    """Rows are months, columns are counties."""
    return (
        panel.pivot(index="DateForSearch", columns="County", values=metric)
        .sort_index()
        .reindex(columns=MN_COUNTIES)   # all 87 counties, Wilkin included
        .fillna(0.0)
    )


def predict_august(w: pd.DataFrame, year: int, method: str) -> pd.Series:
    """County forecast for August of `year`, using only months through that July."""
    july = w.loc[pd.Timestamp(year, 7, 1)]
    june = w.loc[pd.Timestamp(year, 6, 1)]
    may = w.loc[pd.Timestamp(year, 5, 1)]
    if method == "july_only":
        return july
    if method == "last_august":
        return w.loc[pd.Timestamp(year - 1, 8, 1)]
    if method == "may_june_july_average":
        return (may + june + july) / 3
    raise ValueError(method)


METHODS = ["july_only", "last_august", "may_june_july_average"]
pounds = wide("pounds")
actual_2025 = float(pounds.loc[pd.Timestamp("2025-08-01")].sum())

print(f"Actual statewide pounds in August 2025: {actual_2025:,.0f}")
print()
for method in METHODS:
    guessed = float(predict_august(pounds, 2025, method).sum())
    gap = guessed / actual_2025 - 1
    print(f"{method:<24}  guess {guessed:12,.0f}   vs actual {gap:+.1%}")

Actual statewide pounds in August 2025: 12,821,574

july_only                 guess   12,796,557   vs actual -0.2%
last_august               guess   13,235,462   vs actual +3.2%
may_june_july_average     guess   12,208,044   vs actual -4.8%


## Step 5. Score the three forecasts on past Augusts

A good statewide total can still be wrong county by county. One county can be too high and another too low, and the errors cancel in the sum.

Two scores are used:

- **RMSE** (root mean square error) across the 87 counties. This is the score the judges use. It is in the units of the measure, and big counties dominate it: a 5% miss on Hennepin outweighs a 50% miss on twenty small counties.
- **Error share**: add up the absolute misses across counties, then divide by the real August total. An error share of 0.11 means the misses add up to 11% of the pounds (or people, or households). Easier to read than RMSE.

We score August 2023, August 2024, and August 2025. August 2023 and 2024 are the comparison years. August 2025 is the dress rehearsal: it is scored after the method is chosen, and it is not used to pick the winner.

The May–July average has the lowest RMSE on five of the six measures in the dress rehearsal. For seniors, July-only was lower in 2025, but the average was lower in both 2023 and 2024, so the average is kept for seniors too. One rule for all six measures is also easier to explain.


In [6]:
def error_share(actual: pd.Series, pred: pd.Series) -> float:
    a = actual.to_numpy(dtype=float)
    p = pred.reindex(actual.index).to_numpy(dtype=float)
    ok = np.isfinite(a) & np.isfinite(p) & (a > 0)
    return float(np.abs(a[ok] - p[ok]).sum() / a[ok].sum())


def rmse(actual: pd.Series, pred: pd.Series) -> float:
    a = actual.to_numpy(dtype=float)
    p = pred.reindex(actual.index).to_numpy(dtype=float)
    return float(np.sqrt(np.mean((a - p) ** 2)))


METRICS = ["households", "pounds", "individuals", "children", "adults", "seniors"]
records = []
for metric in METRICS:
    w = wide(metric)
    for year in (2023, 2024, 2025):
        actual = w.loc[pd.Timestamp(year, 8, 1)]
        for method in METHODS:
            pred = predict_august(w, year, method)
            records.append({
                "metric": metric,
                "year": year,
                "method": method,
                "rmse": rmse(actual, pred),
                "error_share": error_share(actual, pred),
            })

scores = pd.DataFrame(records)
comparison = (
    scores[scores["year"].isin([2023, 2024])]
    .groupby(["metric", "method"])[["rmse", "error_share"]]
    .mean()
    .rename(columns={"rmse": "rmse_2023_2024", "error_share": "error_share_2023_2024"})
)
dress = (
    scores[scores["year"].eq(2025)]
    .set_index(["metric", "method"])[["rmse", "error_share"]]
    .rename(columns={"rmse": "rmse_2025", "error_share": "error_share_2025"})
)
report = pd.concat([comparison, dress], axis=1).reset_index()
report["metric"] = pd.Categorical(report["metric"], METRICS, ordered=True)
report = report.sort_values(["metric", "rmse_2025"])
print("Lower is closer. RMSE is the judges' score. August 2025 was not used to pick the method.")
display(
    report.style
    .format({"rmse_2023_2024": "{:,.0f}", "rmse_2025": "{:,.0f}", "error_share_2023_2024": "{:.1%}", "error_share_2025": "{:.1%}"})
    .hide(axis="index")
)


Lower is closer. RMSE is the judges' score. August 2025 was not used to pick the method.


metric,method,rmse_2023_2024,error_share_2023_2024,rmse_2025,error_share_2025
households,may_june_july_average,902,9.9%,500,5.6%
households,last_august,"2,170",22.4%,861,9.4%
households,july_only,943,10.7%,"1,102",9.8%
pounds,may_june_july_average,"46,950",9.5%,"61,245",11.1%
pounds,july_only,"52,206",10.9%,"66,761",11.8%
pounds,last_august,"79,759",15.7%,"70,156",14.9%
individuals,may_june_july_average,"3,030",9.3%,575,3.3%
individuals,july_only,"3,282",10.9%,"2,090",7.8%
individuals,last_august,"5,972",20.9%,"2,991",9.3%
children,may_june_july_average,"1,331",11.0%,269,4.1%


**Decision.** Use the May–July average for every measure.

On the dress rehearsal (August 2025) it has the lowest RMSE for visits, pounds, individuals, children, and adults. For seniors it was second to July-only in 2025 and first in 2023 and 2024. In error-share terms it missed 11.1% of pounds, 3.3% of people, and 5.6% of households. Carrying July forward was worse on all of them. Carrying last August forward was worse still, because a county's August jump in one year does not repeat the next year.

One caution for pounds: statewide, August has come in **above** the May–July average in every past year (ratios 1.09, 1.14, 1.00, 1.05). Scaling the average up by that ratio fixes 2023 but makes 2024 and 2025 worse, so the plain average is kept. Treat the pounds forecast as slightly conservative.


## Step 6. Try one public series, then leave it out

The public series is the BLS county unemployment rate. The file `forecast/unemployment_through_july2026.csv` was saved after dropping every month after July 2026, so this notebook does not download anything and does not read an August 2026 rate.

The idea: if a county's unemployment rate is higher than a year ago, food-shelf volume might come in above the May–July average. A single slope was fit on months through July 2024 (about a 5% increase in pounds for a one-point rise in the unemployment rate). That version looked slightly better on 2023 and 2024.

On the August 2025 dress rehearsal it made people and households worse (in error-share terms, 3.3% to 6.4% for people, 5.6% to 8.0% for households). Pounds improved only from 11.1% to 10.4%. The table below is in RMSE, the judges' score, and tells the same story: on RMSE the unemployment version did not even win the 2023–2024 selection for pounds. The published forecast does not use unemployment.

In [7]:
unemp = pd.read_csv("forecast/unemployment_through_july2026.csv", parse_dates=["DateForSearch"])
print(f"Unemployment file: {unemp['County'].nunique()} counties, {unemp['DateForSearch'].min().date()} through {unemp['DateForSearch'].max().date()}")
if unemp["DateForSearch"].max() > pd.Timestamp("2026-07-01"):
    raise RuntimeError("Unemployment file goes past July 2026.")

from forecast_august import score_models

scored, fits = score_models(panel, unemp)
view = scored.pivot_table(index=["metric", "model"], columns="year", values="rmse")
print()
print("RMSE across counties by year. trail3 is the May–July average. trail3_unemp scales it by unemployment.")
display(view.style.format("{:,.0f}"))
print()
for metric, by_year in fits.items():
    fit = by_year[2024]
    print(f"{metric}: slope through July 2024 = {fit['slope']:+.3f} (standard error {fit['se']:.3f})")

Unemployment file: 87 counties, 2022-01-01 through 2026-07-01



RMSE across counties by year. trail3 is the May–July average. trail3_unemp scales it by unemployment.



households: slope through July 2024 = +0.047 (standard error 0.010)
pounds: slope through July 2024 = +0.047 (standard error 0.012)
individuals: slope through July 2024 = +0.061 (standard error 0.009)
children: slope through July 2024 = +0.065 (standard error 0.011)
adults: slope through July 2024 = +0.063 (standard error 0.009)
seniors: slope through July 2024 = +0.052 (standard error 0.013)


## Step 7. Apply the May–July average to August 2026

There is no August 2026 column in the source file. The function below reads May, June, and July 2026 only.

The table written here is the same forecast as `forecast/august_2026_by_county.csv`, and the five submission columns are the same as `forecast/Grad_Team_XX_TFG_MinneMUDAC.xlsx`, which is the organizers' template filled in with their county spelling and column order.

| Column | What it is |
|---|---|
| `households_forecast` | Predicted August 2026 visits (template column **Visits**) |
| `pounds_forecast` | Predicted August 2026 pounds (**Pounds**) |
| `children_forecast`, `adults_forecast`, `seniors_forecast` | Predicted August 2026 age groups (**Children**, **Adults**, **Seniors**) |
| `individuals_forecast` | Predicted people; equals the three age groups added together, within rounding |
| `*_july2026` | Reported July 2026, already in the extract |
| `*_aug2025` | Reported August 2025, already in the extract |

The `_p10` and `_p90` columns in the CSV are a low and high band around the forecast, from how far the method missed in earlier Augusts. They are not a second forecast.


In [8]:
forecast = pd.DataFrame(index=pd.Index(MN_COUNTIES, name="County"))
for metric in METRICS:
    w = wide(metric)
    forecast[f"{metric}_forecast"] = predict_august(w, 2026, "may_june_july_average")
    forecast[f"{metric}_july2026"] = w.loc[pd.Timestamp("2026-07-01")]
    forecast[f"{metric}_aug2025"] = w.loc[pd.Timestamp("2025-08-01")]

forecast = forecast.round(0).astype(int).sort_values("pounds_forecast", ascending=False).reset_index()

print(f"{len(forecast)} counties. Statewide August 2026 forecast")
for label in METRICS:
    print(
        f"  {label:<12} forecast {forecast[f'{label}_forecast'].sum():>12,}    "
        f"July 2026 {forecast[f'{label}_july2026'].sum():>12,}    "
        f"August 2025 {forecast[f'{label}_aug2025'].sum():>12,}"
    )

saved = pd.read_csv("forecast/august_2026_by_county.csv")
check = forecast.merge(saved, on="County", suffixes=("", "_saved"))
mismatch = sum(
    int((check[f"{m}_forecast"] - check[f"{m}_forecast_saved"]).abs().sum()) for m in METRICS
)
print()
print(f"Difference versus forecast/august_2026_by_county.csv, all measures: {mismatch:,}")

submission = pd.read_excel("forecast/Grad_Team_XX_TFG_MinneMUDAC.xlsx")
template = pd.read_excel("../../By Organizers/PredictionFile/Grad_Team_XX_TFG_MinneMUDAC.xlsx")
print(f"Submission rows: {len(submission)}, columns: {list(submission.columns)}")
print("County order and spelling match the template:", submission["County"].tolist() == template["County"].dropna().tolist())
print("Blank cells in submission:", int(submission.isna().sum().sum()))
print()
forecast.head(10)


87 counties. Statewide August 2026 forecast
  households   forecast      236,148    July 2026      239,143    August 2025      240,290
  pounds       forecast   12,263,170    July 2026   12,999,406    August 2025   12,821,573
  individuals  forecast      730,121    July 2026      739,444    August 2025      727,272
  children     forecast      258,020    July 2026      259,559    August 2025      259,670
  adults       forecast      386,891    July 2026      394,311    August 2025      382,478
  seniors      forecast       85,216    July 2026       85,574    August 2025       85,124

Difference versus forecast/august_2026_by_county.csv, all measures: 0


Submission rows: 87, columns: ['County', 'Visits', 'Pounds', 'Children', 'Adults', 'Seniors']
County order and spelling match the template: True
Blank cells in submission: 0



,County,households_forecast,households_july2026,households_aug2025,pounds_forecast,pounds_july2026,pounds_aug2025,individuals_forecast,individuals_july2026,individuals_aug2025,children_forecast,children_july2026,children_aug2025,adults_forecast,adults_july2026,adults_aug2025,seniors_forecast,seniors_july2026,seniors_aug2025
0,Hennepin,70710,69835,80493,3830498,3936204,4071707,217369,218738,231412,79842,79300,85608,117224,118900,119552,20304,20538,26252
1,Ramsey,22510,21898,21354,1093130,1090003,1642390,77943,75631,69334,29965,28560,26018,40933,40346,36767,7045,6725,6549
2,Dakota,14563,15654,12360,876135,988759,882977,43425,48111,40014,15221,17590,14105,22435,24431,21335,5769,6090,4574
3,Anoka,16232,15832,20888,759033,815334,819551,65581,59189,72272,27535,23206,26586,32469,30754,36799,5578,5229,8887
4,Washington,22296,22532,19591,744138,762648,682075,75831,75796,69678,25819,25469,24830,39730,39874,37127,10282,10453,7721
5,Olmsted,10147,11001,10373,495069,604533,437322,28121,30194,28318,7917,8682,7972,16432,17450,16918,3772,4062,3428
6,Stearns,4907,5177,4359,394190,427401,355694,12248,13010,11574,4362,4677,4111,6733,7144,6321,1153,1189,1142
7,St. Louis,7020,6488,6245,379051,384187,324378,17528,16664,15540,4562,4394,4091,9674,9205,9330,3292,3065,2119
8,Blue Earth,4603,5010,5134,271400,290297,273079,14001,15145,15455,6255,6947,7206,6948,7351,7510,797,847,739
9,Rice,4081,4439,4210,224750,240690,208760,13206,14543,13345,5011,5661,5051,6937,7588,7095,1259,1294,1199


In [9]:
history = state.copy()
fig, ax = plt.subplots()
ax.plot(history.index, history["pounds"] / 1e6, color="#1D4E89", lw=2, label="Reported")
forecast_total = forecast["pounds_forecast"].sum() / 1e6
ax.scatter([pd.Timestamp("2026-08-01")], [forecast_total], color="#B42318", s=40, zorder=3, label="August 2026 forecast")
ax.set_title("Statewide pounds, with the August 2026 forecast marked")
ax.set_ylabel("Pounds (millions)")
ax.set_xlabel("Month")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()
print(f"The red point is a prediction: {forecast['pounds_forecast'].sum():,.0f} pounds. It is not a reported August.")

The red point is a prediction: 12,263,170 pounds. It is not a reported August.


/var/folders/jq/pnbcz95j78957hwkvq1nhmpr0000gn/T/ipykernel_86438/2990808251.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Step 8. Two counties where last August was an odd month

Ramsey's forecast is about 1.09 million pounds. August 2025 was 1.64 million. That gap is one site in August 2025 reporting about 785,000 pounds, then returning to about 290,000 in September. The other recent Ramsey months sit near 1.1 million, which is what the average of May–July says.

Itasca's August 2025 was about 16,000 pounds. The months around it are near 90,000. The forecast, about 89,000, follows the recent level.

In [10]:
recent = panel[panel["DateForSearch"] >= "2025-01-01"].copy()
recent["month"] = recent["DateForSearch"].dt.strftime("%Y-%m")
for county in ["Ramsey", "Itasca"]:
    series = recent.loc[recent["County"].eq(county), ["month", "pounds", "individuals"]].set_index("month")
    print(county)
    display(series.style.format("{:,.0f}"))
    print()

Ramsey


,pounds,individuals
month,,
2025-01,"1,099,829","70,640"
2025-02,"1,008,176","68,128"
2025-03,"1,136,177","73,561"
2025-04,"1,132,109","77,343"
2025-05,"1,128,153","70,644"
2025-06,"1,107,303","70,028"
2025-07,"1,083,641","71,021"
2025-08,"1,642,390","69,334"
2025-09,"1,157,812","70,954"



Itasca


,pounds,individuals
month,,
2025-01,"109,716","4,386"
2025-02,"87,789","3,642"
2025-03,"110,765","4,393"
2025-04,"108,487","4,884"
2025-05,"102,916","4,401"
2025-06,"102,916","4,401"
2025-07,"92,375","4,098"
2025-08,"16,002","1,191"
2025-09,"17,706","5,047"


## What to use

Submit `forecast/Grad_Team_XX_TFG_MinneMUDAC.xlsx` (or the `.csv` next to it) after replacing `XX` with the team number. It has 87 rows in the organizers' county spelling and the five columns they asked for: Visits, Pounds, Children, Adults, Seniors.

Statewide August 2026 forecast:

- 236,148 visits (households)
- 12,263,170 pounds
- 258,020 children, 386,891 adults, 85,216 seniors (730,127 people together; the separate individuals forecast is 730,121, the difference is rounding)

Wilkin is in the table with zeros. It has no food-shelf history in the extract.

Two things to say out loud when presenting: the pounds number is likely a little low, since past Augusts have come in above the May–July average; and the single biggest risk to the score is a one-off site anomaly in a large county, like Ramsey in August 2025, which no method based on history can predict.
